# CUDA Fundamentals

Establish the CUDA execution concepts needed for later GPU experiments.

## Objectives

Detect CUDA safely and frame an experiment around host/device execution and synchronization.

## Background

CUDA launches work on a GPU execution hierarchy; asynchronous execution means correct timing requires deliberate synchronization.

## Prediction

A CUDA operation has at least three distinguishable costs:

1. host-side launch overhead;
2. GPU execution time;
3. host-side waiting caused by synchronization.

Because CUDA launches are normally asynchronous, measuring only the Python call duration should initially report mostly host-side enqueue overhead rather than completed GPU work.

The first CUDA operation may be slower than later operations because the CUDA context, memory allocator, and kernel machinery may require one-time initialization.

For very small tensors, fixed launch and synchronization overhead should dominate. As tensor size increases, GPU execution time should become a larger fraction of total elapsed time.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [4]:
from pprint import pprint

from common.cuda import detect_cuda


cuda_info = detect_cuda()
pprint(cuda_info)

if not cuda_info.torch_installed:
    raise RuntimeError(
        "PyTorch is not installed in this environment. "
        "CUDA experiments cannot continue."
    )

if not cuda_info.available:
    raise RuntimeError(
        "PyTorch is installed, but CUDA is not available. "
        f"Detection error: {cuda_info.error!r}"
    )

CudaInfo(torch_installed=True,
         available=True,
         device_count=1,
         device_names=('NVIDIA GB10',),
         torch_version='2.13.0+cu130',
         cuda_version='13.0',
         error=None)


### CUDA runtime and device properties

Before measuring execution, inspect the CUDA runtime exposed through PyTorch and the properties of the selected device.

These values are environment facts. They describe the software and hardware visible to this process but do not yet measure performance.

In [5]:
import torch


device_index = torch.cuda.current_device()
device = torch.device(f"cuda:{device_index}")
properties = torch.cuda.get_device_properties(device_index)

runtime_info = {
    "torch_version": torch.__version__,
    "torch_cuda_build_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "device_count": torch.cuda.device_count(),
    "current_device_index": device_index,
    "current_device_name": torch.cuda.get_device_name(device_index),
    "compute_capability": (
        properties.major,
        properties.minor,
    ),
    "multiprocessor_count": properties.multi_processor_count,
    "total_device_memory_bytes": properties.total_memory,
    "total_device_memory_gib": properties.total_memory / 1024**3,
}

pprint(runtime_info)

{'compute_capability': (12, 1),
 'cuda_available': True,
 'current_device_index': 0,
 'current_device_name': 'NVIDIA GB10',
 'device_count': 1,
 'multiprocessor_count': 48,
 'torch_cuda_build_version': '13.0',
 'torch_version': '2.13.0+cu130',
 'total_device_memory_bytes': 130663002112,
 'total_device_memory_gib': 121.68940353393555}


In [6]:
values = torch.arange(
    8,
    dtype=torch.float32,
    device=device,
)

result = values * 2.0 + 1.0

print(f"values device: {values.device}")
print(f"result device: {result.device}")
print(f"result: {result.cpu().tolist()}")

expected = [1.0, 3.0, 5.0, 7.0, 9.0, 11.0, 13.0, 15.0]
assert result.cpu().tolist() == expected

values device: cuda:0
result device: cuda:0
result: [1.0, 3.0, 5.0, 7.0, 9.0, 11.0, 13.0, 15.0]


### Asynchronous launch and synchronization

A CUDA tensor operation invoked from Python is normally enqueued onto a CUDA stream. The Python call may return before the GPU has completed the operation.

Three timings are compared:

- **enqueue time** measures how long the Python call takes without waiting explicitly;
- **synchronized host time** includes the host-side wait for completion;
- **CUDA event time** measures elapsed time on the CUDA stream around the operation.

The operation is warmed up before measurement so that this comparison focuses on steady-state execution rather than first-use initialization.

In [ ]:
import time

import pandas as pd


ELEMENT_COUNT = 16_777_216
DTYPE = torch.float32

input_values = torch.linspace(
    0.0,
    1.0,
    ELEMENT_COUNT,
    dtype=DTYPE,
    device=device,
)

output_values = torch.empty_like(input_values)


def cuda_multiply_add() -> None:
    torch.add(
        input_values * 1.000_001,
        0.125,
        out=output_values,
    )


for _ in range(10):
    cuda_multiply_add()

torch.cuda.synchronize(device)

sample_indices = torch.tensor(
    [0, ELEMENT_COUNT // 2, ELEMENT_COUNT - 1],
    device=device,
)

sample_input = input_values[sample_indices].cpu()
sample_output = output_values[sample_indices].cpu()
sample_expected = sample_input * 1.000_001 + 0.125

print(f"Element count: {ELEMENT_COUNT:,}")
print(
    f"Tensor size: {input_values.numel() * input_values.element_size() / 1024**2:.1f} MiB"
)
print(f"Input samples: {sample_input.tolist()}")
print(f"Output samples: {sample_output.tolist()}")

torch.testing.assert_close(sample_output, sample_expected)

Element count: 16,777,216
Tensor size: 64.0 MiB
Input samples: [0.0, 0.5, 1.0]
Output samples: [0.125, 0.6250004768371582, 1.1250009536743164]


In [ ]:
REPETITIONS = 100

timing_rows = []

for repetition in range(REPETITIONS):
    torch.cuda.synchronize(device)

    start_ns = time.perf_counter_ns()
    cuda_multiply_add()
    enqueue_end_ns = time.perf_counter_ns()

    torch.cuda.synchronize(device)
    synchronized_end_ns = time.perf_counter_ns()

    timing_rows.append(
        {
            "repetition": repetition,
            "enqueue_us": (enqueue_end_ns - start_ns) / 1_000,
            "synchronized_us": (synchronized_end_ns - start_ns) / 1_000,
        }
    )

host_timings = pd.DataFrame(timing_rows)

host_timings.describe(percentiles=[0.5, 0.9, 0.99])[["enqueue_us", "synchronized_us"]]

,enqueue_us,synchronized_us
count,100.000000,100.00000
mean,9.858280,1247.90552
std,13.864416,53.67291
min,7.488000,1213.58800
50%,8.040000,1225.52400
90%,10.081600,1333.99280
99%,17.115840,1454.91827
max,146.592000,1518.40400


In [ ]:
event_rows = []

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

for repetition in range(REPETITIONS):
    start_event.record()
    cuda_multiply_add()
    end_event.record()

    end_event.synchronize()

    event_rows.append(
        {
            "repetition": repetition,
            "cuda_event_us": start_event.elapsed_time(end_event) * 1_000,
        }
    )

event_timings = pd.DataFrame(event_rows)

event_timings.describe(percentiles=[0.5, 0.9, 0.99])[["cuda_event_us"]]

,cuda_event_us
count,100.000000
mean,1234.907521
std,43.728412
min,1211.071968
50%,1220.607996
90%,1276.796818
99%,1385.494994
max,1504.992008


In [10]:
timing_summary = pd.DataFrame(
    {
        "measurement": [
            "Python call without synchronization",
            "Python call including synchronization",
            "CUDA events",
        ],
        "median_us": [
            host_timings["enqueue_us"].median(),
            host_timings["synchronized_us"].median(),
            event_timings["cuda_event_us"].median(),
        ],
        "minimum_us": [
            host_timings["enqueue_us"].min(),
            host_timings["synchronized_us"].min(),
            event_timings["cuda_event_us"].min(),
        ],
        "p90_us": [
            host_timings["enqueue_us"].quantile(0.90),
            host_timings["synchronized_us"].quantile(0.90),
            event_timings["cuda_event_us"].quantile(0.90),
        ],
    }
)

timing_summary

,measurement,median_us,minimum_us,p90_us
0,Python call without synchronization,8.040000,7.488000,10.081600
1,Python call including synchronization,1225.524000,1213.588000,1333.992800
2,CUDA events,1220.607996,1211.071968,1276.796818


### Fixed overhead and tensor-size scaling

A single elementwise multiplication is measured over increasing tensor sizes.

Input and output tensors are allocated before timing. The measured operation is:

```python
torch.mul(input_values, scale, out=output_values)

In [ ]:
SCALE = 1.000_001
SIZE_REPETITIONS = 100

element_counts = [
    1_024,
    16_384,
    262_144,
    4_194_304,
    16_777_216,
    67_108_864,
]


def measure_cuda_multiply(element_count: int) -> dict[str, float | int]:
    input_tensor = torch.linspace(
        0.0,
        1.0,
        element_count,
        dtype=torch.float32,
        device=device,
    )
    output_tensor = torch.empty_like(input_tensor)

    def multiply() -> None:
        torch.mul(
            input_tensor,
            SCALE,
            out=output_tensor,
        )

    for _ in range(10):
        multiply()

    torch.cuda.synchronize(device)

    enqueue_times_us = []

    for _ in range(SIZE_REPETITIONS):
        torch.cuda.synchronize(device)

        start_ns = time.perf_counter_ns()
        multiply()
        enqueue_end_ns = time.perf_counter_ns()

        enqueue_times_us.append((enqueue_end_ns - start_ns) / 1_000)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    event_times_us = []

    for _ in range(SIZE_REPETITIONS):
        start_event.record()
        multiply()
        end_event.record()

        end_event.synchronize()

        event_times_us.append(start_event.elapsed_time(end_event) * 1_000)

    sample_indices = torch.tensor(
        [0, element_count // 2, element_count - 1],
        device=device,
    )

    sample_input = input_tensor[sample_indices].cpu()
    sample_output = output_tensor[sample_indices].cpu()

    torch.testing.assert_close(
        sample_output,
        sample_input * SCALE,
    )

    bytes_read_and_written = 2 * element_count * input_tensor.element_size()

    return {
        "element_count": element_count,
        "tensor_mib": (element_count * input_tensor.element_size() / 1024**2),
        "enqueue_median_us": float(pd.Series(enqueue_times_us).median()),
        "event_median_us": float(pd.Series(event_times_us).median()),
        "event_minimum_us": float(min(event_times_us)),
        "effective_bandwidth_gib_s": (
            bytes_read_and_written
            / 1024**3
            / (pd.Series(event_times_us).median() / 1_000_000)
        ),
    }


size_rows = [measure_cuda_multiply(element_count) for element_count in element_counts]

size_timings = pd.DataFrame(size_rows)
size_timings

,element_count,tensor_mib,enqueue_median_us,event_median_us,event_minimum_us,effective_bandwidth_gib_s
0,1024,0.003906,3.456,8.000000,7.808000,0.953674
1,16384,0.062500,3.424,8.000000,7.488000,15.258788
2,262144,1.000000,3.392,7.968000,7.584000,245.121105
3,4194304,16.000000,3.376,149.920002,147.615999,208.444501
4,16777216,64.000000,3.600,594.240010,590.848029,210.352716
5,67108864,256.000000,3.728,2371.695995,2360.127926,210.819600


In [ ]:
size_timings.assign(
    event_time_per_element_ns=(
        size_timings["event_median_us"] * 1_000 / size_timings["element_count"]
    ),
    enqueue_fraction=(
        size_timings["enqueue_median_us"] / size_timings["event_median_us"]
    ),
)

,element_count,tensor_mib,enqueue_median_us,event_median_us,event_minimum_us,effective_bandwidth_gib_s,event_time_per_element_ns,enqueue_fraction
0,1024,0.003906,3.456,8.000000,7.808000,0.953674,7.812500,0.432000
1,16384,0.062500,3.424,8.000000,7.488000,15.258788,0.488281,0.428000
2,262144,1.000000,3.392,7.968000,7.584000,245.121105,0.030396,0.425703
3,4194304,16.000000,3.376,149.920002,147.615999,208.444501,0.035744,0.022519
4,16777216,64.000000,3.600,594.240010,590.848029,210.352716,0.035419,0.006058
5,67108864,256.000000,3.728,2371.695995,2360.127926,210.819600,0.035341,0.001572


### Locating the fixed-overhead crossover

The first scaling experiment places the transition between the approximately 8 µs fixed-overhead floor and the linear throughput regime somewhere between 1 MiB and 16 MiB.

Additional tensor sizes are measured within this interval to identify where execution time begins increasing consistently with logical memory traffic.

In [ ]:
crossover_element_counts = [
    262_144,  # 1 MiB
    524_288,  # 2 MiB
    1_048_576,  # 4 MiB
    2_097_152,  # 8 MiB
    3_145_728,  # 12 MiB
    4_194_304,  # 16 MiB
]

crossover_rows = [
    measure_cuda_multiply(element_count) for element_count in crossover_element_counts
]

crossover_timings = pd.DataFrame(crossover_rows)

crossover_timings.assign(
    logical_traffic_mib=2 * crossover_timings["tensor_mib"],
    event_time_per_element_ns=(
        crossover_timings["event_median_us"]
        * 1_000
        / crossover_timings["element_count"]
    ),
)

,element_count,tensor_mib,enqueue_median_us,event_median_us,event_minimum_us,effective_bandwidth_gib_s,logical_traffic_mib,event_time_per_element_ns
0,262144,1.0,3.456,8.000000,7.744000,244.140613,2.0,0.030518
1,524288,2.0,3.408,8.064000,7.712000,484.406006,4.0,0.015381
2,1048576,4.0,3.376,9.728000,8.832000,803.094197,8.0,0.009277
3,2097152,8.0,3.232,19.967999,16.319999,782.502030,16.0,0.009521
4,3145728,12.0,3.248,82.879998,79.871997,282.788376,24.0,0.026347
5,4194304,16.0,3.312,149.600007,147.295997,208.890365,32.0,0.035667


### Hot versus rotating working sets

The preceding size benchmark repeatedly accessed the same input and output tensors. Smaller tensor pairs may therefore have remained resident in GPU cache.

Two access patterns are compared:

- **hot:** every repetition uses the same input and output region;
- **rotating:** repetitions cycle through non-overlapping regions of much larger input and output allocations.

Both patterns execute the same `torch.mul(..., out=...)` operation. Tensor allocation and slice construction occur before timing.

If cache residency explains the high derived throughput for intermediate sizes, the rotating pattern should be substantially slower than the hot pattern for those sizes. For sufficiently large tensors, the two patterns should converge because even the hot tensor pair cannot remain fully cache-resident.

In [ ]:
WORKING_SET_MIB_PER_ARRAY = 256
CACHE_REPETITIONS = 100

cache_element_counts = [
    262_144,  # 1 MiB
    524_288,  # 2 MiB
    1_048_576,  # 4 MiB
    2_097_152,  # 8 MiB
    3_145_728,  # 12 MiB
    4_194_304,  # 16 MiB
]

pool_element_count = (
    WORKING_SET_MIB_PER_ARRAY
    * 1024**2
    // torch.tensor([], dtype=torch.float32).element_size()
)

pool_input = torch.ones(
    pool_element_count,
    dtype=torch.float32,
    device=device,
)
pool_output = torch.empty_like(pool_input)

print(f"Input pool: {pool_input.numel() * pool_input.element_size() / 1024**2:.0f} MiB")
print(
    f"Output pool: {pool_output.numel() * pool_output.element_size() / 1024**2:.0f} MiB"
)

Input pool: 256 MiB
Output pool: 256 MiB


In [ ]:
def measure_access_pattern(
    element_count: int,
    *,
    rotating: bool,
) -> dict[str, float | int | str]:
    region_count = pool_element_count // element_count

    input_regions = [
        pool_input[region_index * element_count : (region_index + 1) * element_count]
        for region_index in range(region_count)
    ]
    output_regions = [
        pool_output[region_index * element_count : (region_index + 1) * element_count]
        for region_index in range(region_count)
    ]

    def multiply(region_index: int) -> None:
        torch.mul(
            input_regions[region_index],
            SCALE,
            out=output_regions[region_index],
        )

    for repetition in range(max(10, region_count)):
        region_index = repetition % region_count if rotating else 0
        multiply(region_index)

    torch.cuda.synchronize(device)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    event_times_us = []

    for repetition in range(CACHE_REPETITIONS):
        region_index = repetition % region_count if rotating else 0

        start_event.record()
        multiply(region_index)
        end_event.record()
        end_event.synchronize()

        event_times_us.append(start_event.elapsed_time(end_event) * 1_000)

    median_us = float(pd.Series(event_times_us).median())
    logical_bytes = 2 * element_count * 4

    return {
        "access_pattern": "rotating" if rotating else "hot",
        "element_count": element_count,
        "tensor_mib": element_count * 4 / 1024**2,
        "region_count": region_count,
        "median_us": median_us,
        "minimum_us": float(min(event_times_us)),
        "effective_bandwidth_gib_s": (
            logical_bytes / 1024**3 / (median_us / 1_000_000)
        ),
    }


cache_rows = []

for element_count in cache_element_counts:
    cache_rows.append(
        measure_access_pattern(
            element_count,
            rotating=False,
        )
    )
    cache_rows.append(
        measure_access_pattern(
            element_count,
            rotating=True,
        )
    )

cache_timings = pd.DataFrame(cache_rows)
cache_timings

,access_pattern,element_count,tensor_mib,region_count,median_us,minimum_us,effective_bandwidth_gib_s
0,hot,262144,1.0,256,8.096000,7.776000,241.245672
1,rotating,262144,1.0,256,15.072000,13.632000,129.586317
2,hot,524288,2.0,128,8.064000,7.776000,484.406006
3,rotating,524288,2.0,128,24.800001,21.792000,157.510076
4,hot,1048576,4.0,64,9.760000,9.536000,800.461081
5,rotating,1048576,4.0,64,43.343998,40.031999,180.244100
6,hot,2097152,8.0,32,21.888001,20.000000,713.861448
7,rotating,2097152,8.0,32,80.240000,75.999998,194.728315
8,hot,3145728,12.0,21,79.264000,78.047998,295.689089
9,rotating,3145728,12.0,21,117.744002,112.608001,199.054725


In [ ]:
cache_comparison = cache_timings.pivot(
    index="tensor_mib",
    columns="access_pattern",
    values=[
        "median_us",
        "effective_bandwidth_gib_s",
    ],
).sort_index()

cache_comparison[("median_us", "rotating_to_hot_ratio")] = (
    cache_comparison[("median_us", "rotating")] / cache_comparison[("median_us", "hot")]
)

cache_comparison

median_us             effective_bandwidth_gib_s              \
access_pattern         hot    rotating                       hot    rotating   
tensor_mib                                                                     
1.0               8.096000   15.072000                241.245672  129.586317   
2.0               8.064000   24.800001                484.406006  157.510076   
4.0               9.760000   43.343998                800.461081  180.244100   
8.0              21.888001   80.240000                713.861448  194.728315   
12.0             79.264000  117.744002                295.689089  199.054725   
16.0            149.728000  154.895999                208.711797  201.748272   

                           median_us  
access_pattern rotating_to_hot_ratio  
tensor_mib                            
1.0                         1.861660  
2.0                         3.075397  
4.0                         4.440984  
8.0                         3.665936  
12.0                        1.485466  
16.0                        1.034516

## Observations

The asynchronous multiply-add experiment processed 16,777,216 FP32 elements. Its median unsynchronized Python duration was 8.040 µs, while synchronized host and CUDA-event timings were approximately 1.23 ms and 1.22 ms respectively.

For the single-operation scaling experiment, median host enqueue time remained nearly constant, between 3.376 µs and 3.728 µs, while tensor size increased from 1,024 to 67,108,864 elements.

CUDA-event duration remained at approximately 8 µs for 1,024, 16,384, and 262,144 elements. The corresponding tensor sizes ranged from 4 KiB to 1 MiB.

For larger tensors, CUDA-event duration increased nearly linearly:

- 16 MiB: 149.920 µs;
- 64 MiB: 594.240 µs;
- 256 MiB: 2,371.696 µs.

The derived effective read-plus-write bandwidth for these three rows was 208.44, 210.35, and 210.82 GiB/s respectively. The derived per-element time stabilized near 0.0354 ns.

## Explanation

The experiments reveal separate host-submission and device-execution behavior.

The host enqueue duration is largely independent of tensor size. Submitting a CUDA operation requires a few microseconds whether the tensor contains one thousand elements or tens of millions. The GPU performs the elementwise work after the host call returns.

For tensors up to 1 MiB, CUDA-event duration remains near an approximately 8 µs floor. In this regime, fixed dispatch, scheduling, event, and minimum-execution costs dominate the useful elementwise work. The experiment does not isolate which portion of this floor belongs to each layer.

For tensors of 16 MiB and larger, execution time scales almost directly with the number of elements. The stable per-element time and effective read-plus-write rate indicate a throughput-limited regime.

The approximately 210 GiB/s effective rate is derived from one four-byte input read and one four-byte output write per element. It is therefore a logical workload throughput, not a direct measurement of physical LPDDR5X traffic or peak memory bandwidth.

The 1 MiB row's higher derived rate should not be interpreted as higher sustained bandwidth. Its execution duration is still constrained by the fixed-overhead floor, which makes bandwidth calculated from elapsed time artificially high.

## Connection to LLMs

LLM kernels rely on CUDA's execution hierarchy, asynchronous launches, and efficient batches of parallel work.

## Further Exploration

TODO: Compare cold-start, warmed-up, synchronized, and unsynchronized timing.